# Medical QLoRA Fine-tuning

Fine-tuning Llama-3.1 8B Instruct on [MedMCQA](https://huggingface.co/datasets/openlifescienceai/medmcqa) (medical entrance exam MCQs) with QLoRA, using Unsloth to fit it on a free Colab T4.

The base model and the fine-tuned model are scored on the same validation questions, so the effect of fine-tuning can be measured directly.

**Runtime:** GPU (T4 works). Training on 5k examples takes roughly 20-25 min.

## Setup

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q unsloth

In [ ]:
import re
import random

import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig

BASE_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
OUTPUT_DIR = "med-qlora-adapter"
MAX_SEQ_LEN = 1024

N_TRAIN = 5000   # MedMCQA has ~183k training questions, a subset is enough to see the effect
N_EVAL = 500

SEED = 42
random.seed(SEED)

## Data

Each MedMCQA row has a question, four options (`opa`-`opd`) and `cop`, the index of the correct option (0-3).
The test split has no labels, so the validation split is used for evaluation.

In [ ]:
ds = load_dataset("openlifescienceai/medmcqa")
print(ds)

train_raw = ds["train"].shuffle(seed=SEED).select(range(N_TRAIN))
eval_raw = ds["validation"].shuffle(seed=SEED).select(range(N_EVAL))

train_raw[0]

In [ ]:
LETTERS = ["A", "B", "C", "D"]
SYSTEM = "You are a medical expert. Answer the multiple choice question with a single letter: A, B, C or D."


def build_question(ex):
    opts = [ex["opa"], ex["opb"], ex["opc"], ex["opd"]]
    lines = [ex["question"].strip(), ""]
    lines += [f"{l}) {o}" for l, o in zip(LETTERS, opts)]
    return "\n".join(lines)


def to_messages(ex, with_answer=True):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": build_question(ex)},
    ]
    if with_answer:
        msgs.append({"role": "assistant", "content": LETTERS[ex["cop"]]})
    return msgs


print(build_question(train_raw[0]))
print("answer:", LETTERS[train_raw[0]["cop"]])

## Load the model in 4-bit

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

## Baseline evaluation

Instead of generating text and parsing it, the model is scored by comparing the logits of the tokens `A`, `B`, `C`, `D` at the position where the answer starts. This is faster and doesn't depend on the output format.

In [ ]:
LETTER_IDS = [tokenizer.encode(l, add_special_tokens=False)[0] for l in LETTERS]


@torch.no_grad()
def evaluate(model, data):
    FastLanguageModel.for_inference(model)
    correct = 0
    for ex in data:
        prompt = tokenizer.apply_chat_template(
            to_messages(ex, with_answer=False), tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
        logits = model(**inputs).logits[0, -1, LETTER_IDS]
        correct += int(logits.argmax().item() == ex["cop"])
    return correct / len(data)


base_acc = evaluate(model, eval_raw)
print(f"Base model accuracy: {base_acc:.1%}")

## Add LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()

## Training

The Llama-3 chat template is applied to every example. `train_on_responses_only` masks the system and user tokens, so the loss is only computed on the answer.

In [ ]:
def to_text(ex):
    return {"text": tokenizer.apply_chat_template(to_messages(ex), tokenize=False)}


train_ds = train_raw.map(to_text, remove_columns=train_raw.column_names)
print(train_ds[0]["text"])

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=10,
        output_dir=OUTPUT_DIR,
        seed=SEED,
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

stats = trainer.train()

In [ ]:
import matplotlib.pyplot as plt

logs = [l for l in trainer.state.log_history if "loss" in l]
plt.plot([l["step"] for l in logs], [l["loss"] for l in logs])
plt.xlabel("step")
plt.ylabel("training loss")
plt.title("Training loss")
plt.show()

## Evaluate the fine-tuned model

In [ ]:
ft_acc = evaluate(model, eval_raw)

print(f"| Model | Accuracy ({N_EVAL} val questions) |")
print("|---|---|")
print(f"| Llama-3.1 8B Instruct (base) | {base_acc:.1%} |")
print(f"| + QLoRA on {N_TRAIN} MedMCQA examples | {ft_acc:.1%} |")

## Try a few questions

In [ ]:
for ex in eval_raw.select(range(5)):
    prompt = tokenizer.apply_chat_template(
        to_messages(ex, with_answer=False), tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**inputs, max_new_tokens=4, do_sample=False)
    pred = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    print(build_question(ex))
    print(f"model: {pred}   correct: {LETTERS[ex['cop']]}")
    print("-" * 60)

## Save the adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
!ls -lh $OUTPUT_DIR

# to push to the Hugging Face Hub:
# model.push_to_hub("your-hf-username/llama3.1-8b-medmcqa-qlora")
# tokenizer.push_to_hub("your-hf-username/llama3.1-8b-medmcqa-qlora")